In [ ]:
# Interoperability: AnnData (Python) ↔ SingleCellExperiment (R)

Single-cell analysis ecosystems exist in both Python (`scanpy` / `AnnData`) and R (`Bioconductor` / `SingleCellExperiment`). The `.h5ad` file format is the on-disk representation of an `AnnData` object and serves as the bridge between the two worlds.

## The `.h5ad` format

An `.h5ad` file is an HDF5 file with a defined layout:

| Slot | AnnData | SingleCellExperiment |
|------|---------|----------------------|
| Count matrix | `adata.X` | `assay(sce, "X")` |
| Cell metadata | `adata.obs` | `colData(sce)` |
| Gene metadata | `adata.var` | `rowData(sce)` |
| Embeddings | `adata.obsm` | `reducedDims(sce)` |
| Unstructured | `adata.uns` | `metadata(sce)` |

## The `zellkonverter` package

[`zellkonverter`](https://bioconductor.org/packages/zellkonverter) is the recommended Bioconductor package for reading and writing `.h5ad` files. It wraps the Python `anndata` library via `basilisk` to ensure faithful round-tripping of all slots.

```r
# install once
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("zellkonverter")
```

## Reading an `.h5ad` file into R

`readH5AD()` is the primary function. It returns a `SingleCellExperiment` object.

In [ ]:
library(zellkonverter)
library(SingleCellExperiment)

# Path to the processed AnnData object saved from Python/scanpy
h5ad_path <- "../data/processed.h5ad"

# Read into a SingleCellExperiment object
sce <- readH5AD(h5ad_path)

sce

## Exploring the `SingleCellExperiment` object

After reading, the slots from `AnnData` are mapped to their SCE equivalents.

In [ ]:
# adata.X  →  assay(sce, "X")  (count / normalized matrix)
dim(assay(sce, "X"))       # genes x cells

# adata.obs  →  colData(sce)  (cell-level metadata)
head(colData(sce))

# adata.var  →  rowData(sce)  (gene-level metadata)
head(rowData(sce))

# adata.obsm  →  reducedDims(sce)  (embeddings: PCA, UMAP, …)
reducedDimNames(sce)

## Accessing specific assays and embeddings

scanpy typically stores the normalized+log-transformed matrix in `adata.layers["log1p"]` and the raw counts in `adata.raw`. `zellkonverter` preserves these as named assays.

In [ ]:
# All available assays (adata.X + any adata.layers)
assayNames(sce)

# Access the log-normalised layer stored by scanpy
log_counts <- assay(sce, "log1p")

# Access the UMAP embedding (adata.obsm["X_umap"])
umap_coords <- reducedDim(sce, "X_umap")
head(umap_coords)

## Plotting the UMAP with Seurat `DimPlot`

Seurat's `DimPlot()` expects a `Seurat` object. We can convert the `SingleCellExperiment` directly using `as.Seurat()`, which maps:

- `assay(sce, "X")` → the default Seurat assay  
- `colData(sce)` → `seurat@meta.data`  
- `reducedDims(sce)` → `seurat@reductions`

In [ ]:
library(Seurat)

# Convert SCE → Seurat
# data = the assay name to use as the Seurat data slot
seurat_obj <- as.Seurat(sce, counts = "X", data = NULL)

# as.Seurat copies reducedDims, so X_umap is already registered
# as a reduction — rename it to the conventional "umap" name
seurat_obj[["umap"]] <- seurat_obj[["X_umap"]]

# Plot — group.by maps to a meta.data column (e.g. leiden, cell_type)
DimPlot(seurat_obj, reduction = "umap", group.by = "leiden", pt.size = 0.5)

## Performance tip: delayed loading with `HDF5Array`

For large datasets, pass `use_hdf5 = TRUE` to keep the matrix on disk as a `HDF5Array` instead of loading it entirely into RAM. Arithmetic operations are still possible via `DelayedArray`.

In [ ]:
# Keep the matrix on disk — useful for datasets > a few GB
sce_hdf5 <- readH5AD(h5ad_path, use_hdf5 = TRUE)

# The assay is now a HDF5Matrix (on-disk)
class(assay(sce_hdf5, "X"))

## Writing a `SingleCellExperiment` back to `.h5ad`

`writeH5AD()` performs the reverse conversion, producing an `.h5ad` readable by scanpy.

In [ ]:
# Write SCE back out as an h5ad file readable by scanpy / Python
writeH5AD(sce, file = "../data/processed_from_R.h5ad")